In [1]:
import gym
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque

# ----------------------
# Hyperparamètres
# ----------------------
GAMMA = 0.99        # facteur de discount
LR = 1e-4           # taux d'apprentissage
BATCH_SIZE = 32
MEMORY_SIZE = 10000
EPS_START = 1.0
EPS_END = 0.01
EPS_DECAY = 50000

# ----------------------
# Définition du réseau
# ----------------------
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, output_dim)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

# ----------------------
# Replay Buffer
# ----------------------
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = map(np.array, zip(*batch))
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)

# ----------------------
# Fonction d'entraînement
# ----------------------
def train_dqn(env_name="CartPole-v1", episodes=500):
    env = gym.make(env_name)
    n_actions = env.action_space.n
    state_dim = env.observation_space.shape[0]

    policy_net = DQN(state_dim, n_actions)
    target_net = DQN(state_dim, n_actions)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(policy_net.parameters(), lr=LR)
    memory = ReplayBuffer(MEMORY_SIZE)
    
    epsilon = EPS_START
    steps_done = 0

    for ep in range(episodes):
        state = env.reset()
        total_reward = 0
        done = False

        while not done:
            steps_done += 1
            # ε-greedy action selection
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action = policy_net(torch.FloatTensor(state)).argmax().item()
            
            next_state, reward, done, _ = env.step(action)
            memory.push(state, action, reward, next_state, done)
            state = next_state
            total_reward += reward
            
            # réduire epsilon
            epsilon = EPS_END + (EPS_START - EPS_END) * np.exp(-1. * steps_done / EPS_DECAY)
            
            # entraîner le réseau
            if len(memory) > BATCH_SIZE:
                states, actions, rewards, next_states, dones = memory.sample(BATCH_SIZE)

                states = torch.FloatTensor(states)
                actions = torch.LongTensor(actions)
                rewards = torch.FloatTensor(rewards)
                next_states = torch.FloatTensor(next_states)
                dones = torch.FloatTensor(dones)

                q_values = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    next_q_values = target_net(next_states).max(1)[0]
                    target_q_values = rewards + GAMMA * next_q_values * (1 - dones)

                loss = nn.MSELoss()(q_values, target_q_values)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            
        # mettre à jour le réseau cible périodiquement
        if ep % 10 == 0:
            target_net.load_state_dict(policy_net.state_dict())
        
        print(f"Episode {ep}, Reward: {total_reward:.2f}, Epsilon: {epsilon:.2f}")

    env.close()
    return policy_net

# ----------------------
# Lancer l'entraînement
# ----------------------
if __name__ == "__main__":
    trained_net = train_dqn("CartPole-v1", episodes=200)


ModuleNotFoundError: No module named 'gym'